In [12]:
import matplotlib.pyplot as plt
import numpy as np

x = np.linspace(-6, 6, 1000)
y = np.sinc(x)

plt.plot(x, y, color="darkblue")
plt.axhline(0, color="black", linewidth=0.5)
plt.axvline(0, color="black", linewidth=0.5)
plt.grid(True, alpha=0.3)

In [18]:
!rm -r /mnt/origin-assn/dataset1_yolo
!unzip -o Drywall-Join-Detect.v2i.yolo26.zip -d /mnt/origin-assn/dataset1_yolo

Archive:  Drywall-Join-Detect.v2i.yolo26.zip
  inflating: /mnt/origin-assn/dataset1_yolo/README.dataset.txt  
  inflating: /mnt/origin-assn/dataset1_yolo/README.roboflow.txt  
  inflating: /mnt/origin-assn/dataset1_yolo/data.yaml  
   creating: /mnt/origin-assn/dataset1_yolo/train/
   creating: /mnt/origin-assn/dataset1_yolo/train/images/
 extracting: /mnt/origin-assn/dataset1_yolo/train/images/2000x1500_0_resized_jpg.rf.0dd5a8210e3178cb5374e1bd32333ff1.jpg  
 extracting: /mnt/origin-assn/dataset1_yolo/train/images/2000x1500_0_resized_jpg.rf.38d4a9203529cd5ab92e28eced776ef7.jpg  
 extracting: /mnt/origin-assn/dataset1_yolo/train/images/2000x1500_0_resized_jpg.rf.d3e86fed1273baffa7b16d27ba3e6c26.jpg  
 extracting: /mnt/origin-assn/dataset1_yolo/train/images/2000x1500_0_resized_jpg.rf.e2838d0ac75a0771cd4a1b1f95706688.jpg  
 extracting: /mnt/origin-assn/dataset1_yolo/train/images/2000x1500_0_resized_jpg.rf.f4b8d1eee966480206093b0834d1d7da.jpg  
 extracting: /mnt/origin-assn/dataset1_yolo/

In [11]:
!rm -r /mnt/origin-assn/dataset2_yolo
!unzip -o crack.v6i.yolo26.zip -d /mnt/origin-assn/dataset2_yolo

Archive:  crack.v6i.yolo26.zip
  inflating: /mnt/origin-assn/dataset2_yolo/README.dataset.txt  
  inflating: /mnt/origin-assn/dataset2_yolo/README.roboflow.txt  
  inflating: /mnt/origin-assn/dataset2_yolo/data.yaml  
   creating: /mnt/origin-assn/dataset2_yolo/test/
   creating: /mnt/origin-assn/dataset2_yolo/test/images/
 extracting: /mnt/origin-assn/dataset2_yolo/test/images/crack_322_jpg.rf.8ab843ad02c57e78d4114d5584d3ba50.jpg  
 extracting: /mnt/origin-assn/dataset2_yolo/test/images/crack_333_jpg.rf.6ed0b1db664dc4bf8fe13f8b33b2eedf.jpg  
 extracting: /mnt/origin-assn/dataset2_yolo/test/images/crack_343_jpg.rf.5c5a8cd8b8aba645b9f89b992b10fe0a.jpg  
 extracting: /mnt/origin-assn/dataset2_yolo/test/images/crack_344_jpg.rf.694389c4d44180047b1cfb959d107647.jpg  
 extracting: /mnt/origin-assn/dataset2_yolo/test/images/crack_349_jpg.rf.0ead8301b4013595355a57d6731c0810.jpg  
 extracting: /mnt/origin-assn/dataset2_yolo/test/images/crack_356_jpg.rf.a80b4c2dfd65d9dfef70dd9207fa547b.jpg  
 ex

In [ ]:
!ls -R /mnt/origin-assn/dataset1_yolo | grep ":$" | sed -e 's/:$//' -e 's/[^-][^\/]*\//--/g' -e 's/^/   /' -e 's/-/|/'
!ls -R /mnt/origin-assn/dataset2_yolo | grep ":$" | sed -e 's/:$//' -e 's/[^-][^\/]*\//--/g' -e 's/^/   /' -e 's/-/|/'

In [1]:
import os
from pathlib import Path

base_dir = Path("/mnt/origin-assn")
ds2_dir = base_dir / "dataset2_yolo"

# Update Dataset 2 labels: Change class ID 0 -> 1
# YOLO label format: class_id x_center y_center width height
print("Updating dataset2 labels (0 -> 1)...")
for label_path in ds2_dir.rglob("labels/*.txt"):
    with open(label_path, "r+") as f:
        lines = f.readlines()
        f.seek(0)
        new_lines = []
        for line in lines:
            parts = line.strip().split()
            if parts and parts[0] == "0":
                parts[0] = "1"
            new_lines.append(" ".join(parts) + "\n")
        f.writelines(new_lines)
        f.truncate()



Updating dataset2 labels (0 -> 1)...
Updated labels and created /mnt/origin-assn/combined_datasets.yaml


In [10]:
import os
from pathlib import Path
import shutil

def convert_yolo_det_to_seg(labels_dir, output_dir):
    """
    Converts YOLO detection labels (cx, cy, w, h) to YOLO segmentation labels (x1, y1, ... x4, y4).
    """
    labels_path = Path(labels_dir)
    output_path = Path(output_dir)
    
    for split in ['train', 'valid', 'test']:
        split_dir = labels_path / split / 'labels'
        out_split_dir = output_path / split / 'labels'
        
        if not split_dir.exists():
            continue
            
        out_split_dir.mkdir(parents=True, exist_ok=True)
        
        for txt_file in split_dir.glob("*.txt"):
            lines_out = []
            
            with open(txt_file, 'r') as f:
                for line in f:
                    parts = line.strip().split()
                    
                    # Standard Detection Format: class, cx, cy, w, h
                    if len(parts) == 5:
                        cls, cx, cy, w, h = map(float, parts)
                        
                        # Convert center/width to polygon corners (top-left, top-right, bottom-right, bottom-left)
                        x1 = cx - w / 2
                        y1 = cy - h / 2
                        x2 = cx + w / 2
                        y2 = cy - h / 2
                        x3 = cx + w / 2
                        y3 = cy + h / 2
                        x4 = cx - w / 2
                        y4 = cy + h / 2
                        
                        new_line = f"{int(cls)} {x1} {y1} {x2} {y2} {x3} {y3} {x4} {y4}\n"
                        lines_out.append(new_line)
                    
                    else:
                        lines_out.append(line)
            
            # Write to new file
            with open(out_split_dir / txt_file.name, 'w') as f:
                f.writelines(lines_out)
                
    # Copy images folder structure (symlink or copy to save space)
    print("Creating symlinks for images...")
    for split in ['train', 'valid', 'test']:
        src_img = labels_path / split / 'images'
        dst_img = output_path / split / 'images'
        if src_img.exists() and not dst_img.exists():
            shutil.copytree(src_img, dst_img)

    print(f"Converted dataset saved to {output_path}")

# Run conversion
convert_yolo_det_to_seg(
    labels_dir="/mnt/origin-assn/dataset1_yolo", 
    output_dir="/mnt/origin-assn/dataset1_yolo_seg"
)

Creating symlinks for images...
Converted dataset saved to /mnt/origin-assn/dataset1_yolo_seg


In [11]:
%%writefile /mnt/origin-assn/combined_datasets.yaml
path: /mnt/origin-assn  # root dir
train:
  - dataset1_yolo_seg/train/images
  - dataset2_yolo/train/images
val:
  - dataset1_yolo_seg/valid/images
  - dataset2_yolo/valid/images
test:
  - dataset2_yolo/test/images

nc: 2
names: ['drywall taping area', 'wall crack']

Overwriting /mnt/origin-assn/combined_datasets.yaml


In [2]:
!wget -q https://docs-assets.developer.apple.com/ml-research/datasets/mobileclip/mobileclip_blt.pt
!wget -q https://github.com/ultralytics/assets/releases/download/v8.4.0/yoloe-26s-seg.pt

In [14]:
!pip uninstall torch torchaudio torchvision -y
%uv pip install torch==2.8.0 torchvision torchaudio -q

Found existing installation: torch 2.10.0
Uninstalling torch-2.10.0:
  Successfully uninstalled torch-2.10.0
Found existing installation: torchaudio 2.10.0
Uninstalling torchaudio-2.10.0:
  Successfully uninstalled torchaudio-2.10.0
Found existing installation: torchvision 0.25.0
Uninstalling torchvision-0.25.0:
  Successfully uninstalled torchvision-0.25.0
Note: you may need to restart the kernel to use updated packages.


In [3]:
!pip uninstall ultralytics -y
!pip install -q git+https://github.com/THU-MIG/yoloe.git
!pip install -q git+https://github.com/THU-MIG/yoloe.git#subdirectory=third_party/CLIP
!pip install -q git+https://github.com/THU-MIG/yoloe.git#subdirectory=third_party/ml-mobileclip
!pip install -q git+https://github.com/THU-MIG/yoloe.git#subdirectory=third_party/lvis-api

from ultralytics import YOLOE
from ultralytics.models.yolo.yoloe.train_pe import YOLOEPESegTrainer
import os
import torch
from ultralytics.utils import yaml_load

os.environ["PYTHONHASHSEED"] = "42"

data = "/mnt/origin-assn/combined_datasets.yaml"
pe_path = "/mnt/origin-assn/fault-pe.pt"
USE_LINEAR_PROBING = True
lr0 = 1e-3  # Learning rate

model = YOLOE("yoloe-11s-seg.pt")

if hasattr(model.model.model[-1], 'savpe'):
    del model.model.model[-1].savpe

model.eval()

names = yaml_load(data)['names']
tpe = model.get_text_pe(names)
torch.save({"names": names, "pe": tpe}, pe_path)

if USE_LINEAR_PROBING:
    head_index = len(model.model.model) - 1
    freeze = [str(f) for f in range(0, head_index)]
    for name, child in model.model.model[-1].named_children():
        if 'cv3' not in name:
            freeze.append(f"{head_index}.{name}")
    freeze.extend([f"{head_index}.cv3.0.0", f"{head_index}.cv3.0.1", f"{head_index}.cv3.1.0", f"{head_index}.cv3.1.1", f"{head_index}.cv3.2.0", f"{head_index}.cv3.2.1"])
else:
    freeze = None

results = model.train(
    data=data,
    epochs=80,
    patience=10,
    batch=16,
    optimizer='AdamW',
    lr0=lr0,
    warmup_bias_lr=0.0,
    weight_decay=0.025,
    momentum=0.9,
    workers=4,
    val_interval=1,
    project="/mnt/origin-assn/runs/segment",
    name="v11-construction_segmentation",
    trainer=YOLOEPESegTrainer,
    freeze=freeze,
    train_pe_path=pe_path,
    close_mosaic=5
)

Found existing installation: ultralytics 8.3.39
Uninstalling ultralytics-8.3.39:
  Successfully uninstalled ultralytics-8.3.39

[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Build text model mobileclip:blt
New https://pypi.org/project/ultralytics/8.4.21 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.39 🚀 Python-3.12.6 torch-2.8.0+cu129 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: task=segment, mode=train, model=yoloe-11s-seg.pt, data=/mnt/origin-assn/combined_datasets.yaml, epochs=80, time=None, patience=10, batch=16, imgsz=640, save=True, save_period=-1, cache=False,

100%|███████████████████████████████████████████████████████████████████████| 755k/755k [00:00<00:00, 30.2MB/s]


Overriding model.yaml nc=80 with nc=2

                   from  n    params  module                                       arguments                     
  0                  -1  1       928  ultralytics.nn.modules.conv.Conv             [3, 32, 3, 2]                 
  1                  -1  1     18560  ultralytics.nn.modules.conv.Conv             [32, 64, 3, 2]                
  2                  -1  1     26080  ultralytics.nn.modules.block.C3k2            [64, 128, 1, False, 0.25]     
  3                  -1  1    147712  ultralytics.nn.modules.conv.Conv             [128, 128, 3, 2]              
  4                  -1  1    103360  ultralytics.nn.modules.block.C3k2            [128, 256, 1, False, 0.25]    
  5                  -1  1    590336  ultralytics.nn.modules.conv.Conv             [256, 256, 3, 2]              
  6                  -1  1    346112  ultralytics.nn.modules.block.C3k2            [256, 256, 1, True]           
  7                  -1  1   1180672  ultralytics

100%|█████████████████████████████████████████████████████████████████████| 5.35M/5.35M [00:00<00:00, 80.7MB/s]


AMP: checks passed ✅


...

      71/80      1.34G      1.849      8.504      2.368      1.894         14        640: 100%|██████████| 91/9
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R 



                   all        262        321      0.334      0.275      0.199     0.0753      0.203      0.178      0.113      0.036
EarlyStopping: Training stopped early as no improvement observed in last 10 epochs. Best results observed at epoch 61, best model saved as best.pt.
To update EarlyStopping(patience=10) pass a new patience value, i.e. `patience=300` or use `patience=0` to disable EarlyStopping.



71 epochs completed in 0.339 hours.
Optimizer stripped from /mnt/origin-assn/runs/segment/v11-construction_segmentation/weights/last.pt, 20.5MB
Optimizer stripped from /mnt/origin-assn/runs/segment/v11-construction_segmentation/weights/best.pt, 20.5MB

Validating /mnt/origin-assn/runs/segment/v11-construction_segmentation/weights/best.pt...
Ultralytics 8.3.39 🚀 Python-3.12.6 torch-2.8.0+cu129 CUDA:0 (Tesla T4, 14913MiB)
YOLOe-11s-seg summary (fused): 270 layers, 10,067,590 parameters, 0 gradients, 32.8 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R 




                   all        262        321      0.336      0.275      0.202     0.0764      0.205      0.178      0.113      0.037
   drywall taping area        202        250      0.347      0.128      0.127     0.0377     0.0357      0.004    0.00216   0.000405
            wall crack         60         71      0.326      0.423      0.277      0.115      0.375      0.352      0.223     0.0736
Speed: 0.3ms preprocess, 5.7ms inference, 0.0ms loss, 1.3ms postprocess per image
Results saved to /mnt/origin-assn/runs/segment/v11-construction_segmentation


In [1]:
!pip uninstall ultralytics -y
!pip install -U ultralytics -q
from ultralytics import YOLOE
from ultralytics.models.yolo.yoloe import YOLOEPESegTrainer
import os

model = YOLOE("yoloe-26s-seg.pt")

# This also trains based on text embeddings, refer: https://github.com/ultralytics/ultralytics/blob/7710ef05dc56dfe61b20a7537f17078db2b7170b/ultralytics/models/yolo/yoloe/train.py#L144
results = model.train(
    data="/mnt/origin-assn/combined_datasets.yaml",  # Segmentation dataset
    epochs=80,
    patience=10,
    trainer=YOLOEPESegTrainer,  # <- Important: use segmentation trainer
    save_dir="/mnt/origin-assn/runs/segment/v26_fault_segmentation"
)



[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
WARNING ⚠️ user config directory '/root/.config/Ultralytics' is not writable, using '/tmp/Ultralytics'. Set YOLO_CONFIG_DIR to override.
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/tmp/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.21 🚀 Python-3.12.6 torch-2.8.0+cu129 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/mnt/origin-assn/combined_datasets.yaml, degrees=0.0, deterministic=True, device=No

In [19]:
%%writefile eval.py
"""
YOLOE Segmentation Evaluation — IoU & Dice
Usage:
    python eval.py --model runs/segment/best.pt --data combined_datasets.yaml --split val
    python eval.py --model runs/segment/best.pt --data combined_datasets.yaml --split test
"""

import argparse
import numpy as np
from pathlib import Path
from ultralytics import YOLOE
from ultralytics.utils import YAML
import torch
import cv2
from tqdm import tqdm

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)


def compute_metrics(pred_mask, gt_mask):
    """Return IoU and Dice for a single binary mask pair."""
    pred = pred_mask.astype(bool)
    gt   = gt_mask.astype(bool)
    intersection = (pred & gt).sum()
    union        = (pred | gt).sum()
    iou  = intersection / union  if union  > 0 else float("nan")
    dice = 2 * intersection / (pred.sum() + gt.sum()) if (pred.sum() + gt.sum()) > 0 else float("nan")
    return iou, dice


def load_split_pairs(data_cfg, split):
    """Return list of (image_path, label_path) for the requested split."""
    cfg  = YAML.load(data_cfg)
    root = Path(cfg["path"])
    sources = cfg[split]
    if isinstance(sources, str):
        sources = [sources]

    pairs = []
    for img_dir in sources:
        img_dir = root / img_dir
        lbl_dir = Path(str(img_dir).replace("images", "labels"))
        for img_path in sorted(img_dir.glob("*.*")):
            if img_path.suffix.lower() not in {".jpg", ".jpeg", ".png", ".bmp"}:
                continue
            lbl_path = lbl_dir / (img_path.stem + ".txt")
            if lbl_path.exists():
                pairs.append((img_path, lbl_path))
    return pairs


def run_eval(model_path, data_cfg, split, conf=0.25):
    cfg   = YAML.load(data_cfg)
    names = cfg["names"]   # ['drywall taping area', 'wall crack']
    pairs = load_split_pairs(data_cfg, split)
    print(f"\n[{split}] {len(pairs)} images | classes: {names}\n")

    model = YOLOE(model_path)
    model.set_classes(['drywall seam', 'crack'])
    model.eval()

    stats = {cls: {"iou": [], "dice": []} for cls in names}

    for img_path, lbl_path in tqdm(pairs, desc=split):
        img  = cv2.imread(str(img_path))
        h, w = img.shape[:2]

        gt_masks = {cls: np.zeros((h, w), dtype=np.uint8) for cls in names}
        with open(lbl_path) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) < 5:
                    continue
                cls_id = int(parts[0])
                pts    = (np.array(parts[1:], dtype=np.float32)
                          .reshape(-1, 2) * np.array([w, h])).astype(np.int32)
                cv2.fillPoly(gt_masks[names[cls_id]], [pts], 1)

        results = model.predict(img, conf=conf, verbose=False)
        result  = results[0]

        pred_masks = {cls: np.zeros((h, w), dtype=np.uint8) for cls in names}
        if result.masks is not None:
            for m, c in zip(result.masks.data.cpu().numpy(),
                            result.boxes.cls.cpu().numpy().astype(int)):
                m_resized = cv2.resize(m, (w, h), interpolation=cv2.INTER_NEAREST)
                pred_masks[names[c]] = np.maximum(
                    pred_masks[names[c]], (m_resized > 0.5).astype(np.uint8)
                )

        for cls_name in names:
            iou, dice = compute_metrics(pred_masks[cls_name], gt_masks[cls_name])
            stats[cls_name]["iou"].append(iou)
            stats[cls_name]["dice"].append(dice)

    print(f"\n{'Prompt':<30} {'mIoU':>8} {'Dice':>8}")
    print("-" * 48)
    all_iou, all_dice = [], []
    for cls_name in names:
        iou_vals  = [v for v in stats[cls_name]["iou"]  if not np.isnan(v)]
        dice_vals = [v for v in stats[cls_name]["dice"] if not np.isnan(v)]
        miou  = np.mean(iou_vals)  if iou_vals  else float("nan")
        mdice = np.mean(dice_vals) if dice_vals else float("nan")
        all_iou.append(miou)
        all_dice.append(mdice)
        print(f"{'segment ' + cls_name:<30} {miou:>8.4f} {mdice:>8.4f}")

    print("-" * 48)
    print(f"{'Mean':<30} {np.nanmean(all_iou):>8.4f} {np.nanmean(all_dice):>8.4f}\n")


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--model", required=True, help="Path to trained .pt file")
    parser.add_argument("--data",  required=True, help="Path to combined_datasets.yaml")
    parser.add_argument("--split", default="val", choices=["val", "test", "both"])
    parser.add_argument("--conf",  default=0.25,  type=float)
    args = parser.parse_args()

    splits = ["val", "test"] if args.split == "both" else [args.split]
    for s in splits:
        run_eval(args.model, args.data, s, conf=args.conf)

Overwriting eval.py


In [20]:
!python eval.py --model yoloe-26s-seg.pt --data /mnt/origin-assn/combined_datasets.yaml --split val
!python eval.py --model /mnt/origin-assn/runs/segment/v26_fault_segmentation/weights/best.pt --data /mnt/origin-assn/combined_datasets.yaml --split val


[val] 262 images | classes: ['drywall taping area', 'wall crack']

val: 100%|████████████████████████████████████| 262/262 [00:09<00:00, 26.80it/s]

Prompt                             mIoU     Dice
------------------------------------------------
segment drywall taping area      0.0860   0.1389
segment wall crack               0.0000   0.0000
------------------------------------------------
Mean                             0.0430   0.0695


[val] 262 images | classes: ['drywall taping area', 'wall crack']

val: 100%|████████████████████████████████████| 262/262 [00:09<00:00, 27.35it/s]

Prompt                             mIoU     Dice
------------------------------------------------
segment drywall taping area      0.7724   0.8489
segment wall crack               0.5342   0.6789
------------------------------------------------
Mean                             0.6533   0.7639



In [22]:
!mkdir -p pred
import cv2
import random
import matplotlib.pyplot as plt
from pathlib import Path
from ultralytics import YOLOE

model = YOLOE("/mnt/origin-assn/runs/segment/v26_fault_segmentation/weights/best.pt")
model.set_classes(["drywall seam", "wall crack"])


images = (
    random.sample(list(Path("/mnt/origin-assn/dataset1_yolo_seg/valid/images").glob("*.jpg")), 2) +
    random.sample(list(Path("/mnt/origin-assn/dataset2_yolo/valid/images").glob("*.jpg")), 2)
)

for img_path in images:
    result = model.predict(str(img_path), verbose=False)[0]
    h, w   = result.orig_shape
    mask   = np.zeros((h, w), dtype=np.uint8)
    if result.masks is not None:
        for m in result.masks.data.cpu().numpy():
            mask = np.maximum(mask, (cv2.resize(m, (w, h)) > 0.5).astype(np.uint8) * 255)
    cv2.imwrite(f"pred/{img_path.stem}__pred_mask.png", mask)
    cv2.imwrite(f"pred/{img_path.stem}__original.png", cv2.imread(img_path))